# PRISM-GAN — Notebook 01: Acquire DeepFashion-MultiModal and store it on Google Drive

**What this notebook does**
1. Mounts Google Drive and confirms it can actually write to it.
2. Downloads the official DF-MM files (Google Drive IDs from the official README, github.com/yumingj/DeepFashion-MultiModal) to **Colab's local disk**.
3. Verifies every file: zip integrity (CRC check of every member) and member counts against the README (44,096 images, 12,701 parsing masks, 3 label files).
4. Records SHA-256 checksums in `checksums.json`.
5. Copies the verified files to Drive and flushes Drive so the upload completes.
6. **Section B** — run in a *fresh* runtime: re-reads the files from Drive and checks the checksums. Only when Section B reports all PASS is the dataset considered stored.

**Design decision:** Drive stores only the original zips (+ captions JSON), never 50k extracted files. Drive handles tens of thousands of small files poorly (incomplete listings, lost writes) — the cause of the earlier failures. Every later notebook extracts the zips to local disk at start-up.

**If the Drive mount fails with "credential propagation was unsuccessful":** the sign-in popup was not completed. Allow pop-ups for colab.research.google.com, make sure the browser's signed-in Google account is the one you want to use, run the cell again and tick **all** permission boxes (or use "Select all"). If it still fails: *Runtime → Disconnect and delete runtime*, then retry, or click the folder icon in the left sidebar → **Mount Drive**.

**Licence:** DF-MM is for non-commercial research only; do not redistribute images or derived data.

# Section A — download, verify, store (run once)

## A0. Configuration

In [1]:
import os
DRIVE_DIR = "/content/drive/MyDrive/hari/PRISM-GAN/datasets/DeepFashion-MultiModal/raw"   # final home of the zips
LOCAL_DL = "/content/dfmm_download"                                                          # local staging
GET_DENSEPOSE = False     # ~5.6 GB, not needed for PRISM-GAN

# name: (Google Drive file id, approx size from README, check)
FILES = {
    "image.zip":     ("1U2PljA7NE57jcSSzPs21ZurdIPXdYZtN", "~5.4 GB", {"ext": ".jpg", "count": 44096}),
    "parsing.zip":   ("1r-5t-VgDaAQidZLVgWtguaG7DvMoyUv9", "~90 MB",  {"ext": ".png", "count": 12701}),
    "labels.zip":    ("11WoM5ZFwWpVjrIvZajW0g8EmQCNKMAWH", "575 KB",  {"ext": ".txt", "count": 3}),
    "keypoints.zip": ("1ZXdOQI-d4zNhqRJdUEWSQvPwAtLdjovo", "956 KB",  {"ext": ".txt", "count": 2}),
    "captions.json": ("1d1TRm8UMcQhZCb6HpPo8l3OPEin4Ztk2", "~11 MB",  {"json": True}),
}
if GET_DENSEPOSE:
    FILES["densepose.zip"] = ("14uyqBUDDcL1VLaXm7qmqghdcbkFuQa1s", "~5.6 GB", {"ext": ".png", "count": 44096})
SHARED_FOLDER = "https://drive.google.com/drive/folders/1An2c_ZCkeGmhJg0zUjtZF46vyJgQwIr2"
os.makedirs(LOCAL_DL, exist_ok=True)

## A1. Mount Drive and prove it is writable

In [2]:
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)
probe = os.path.join(DRIVE_DIR, ".write_probe")
with open(probe, "w") as f: f.write("ok")
assert open(probe).read() == "ok"; os.remove(probe)
import shutil
tot, used, free = shutil.disk_usage("/content/drive/MyDrive")
print(f"Drive mounted and writable: {DRIVE_DIR}")
print(f"Drive reports ~{free / 1e9:.1f} GB free (need ~5.6 GB, ~11.2 GB with DensePose)")
print(f"Local disk free: {shutil.disk_usage('/content').free / 1e9:.1f} GB")

Mounted at /content/drive
Drive mounted and writable: /content/drive/MyDrive/hari/PRISM-GAN/datasets/DeepFashion-MultiModal/raw
Drive reports ~3.8 GB free (need ~5.6 GB, ~11.2 GB with DensePose)
Local disk free: 220.6 GB


## A2. Download to local disk (with retries)

In [3]:
import subprocess, time, zipfile, json
subprocess.run(["pip", "install", "-q", "-U", "gdown"], check=True)
import gdown

def is_valid_quick(path, spec):
    if not os.path.exists(path) or os.path.getsize(path) == 0: return False
    if spec.get("json"):
        try: json.load(open(path)); return True
        except Exception: return False
    return zipfile.is_zipfile(path)

failed = []
for name, (fid, approx, spec) in FILES.items():
    dst = os.path.join(LOCAL_DL, name)
    if is_valid_quick(dst, spec):
        print(f"[skip] {name} already downloaded"); continue
    for attempt in range(1, 4):
        print(f"[get ] {name} ({approx}) attempt {attempt}")
        try:
            gdown.download(id=fid, output=dst, quiet=False)
        except Exception as e:
            print("       error:", e)
        if is_valid_quick(dst, spec): break
        if os.path.exists(dst): os.remove(dst)
        time.sleep(20 * attempt)
    else:
        failed.append(name)

if failed:
    raise RuntimeError(
        f"Download failed for {failed}. Usually a Google Drive download quota on the shared file. "
        f"Options: wait a few hours and rerun this cell (finished files are skipped), or open {SHARED_FOLDER}, "
        f"download the missing files in the browser and upload them into {LOCAL_DL} (or into DRIVE_DIR and copy).")
print("all downloads present")

[get ] image.zip (~5.4 GB) attempt 1


Downloading...
From (original): https://drive.google.com/uc?id=1U2PljA7NE57jcSSzPs21ZurdIPXdYZtN
From (redirected): https://drive.google.com/uc?id=1U2PljA7NE57jcSSzPs21ZurdIPXdYZtN&confirm=t&uuid=88f73f9f-875d-4b84-88d8-c9c088de8926
To: /content/dfmm_download/image.zip
100%|██████████| 6.82G/6.82G [01:00<00:00, 112MB/s]


[get ] parsing.zip (~90 MB) attempt 1


Downloading...
From (original): https://drive.google.com/uc?id=1r-5t-VgDaAQidZLVgWtguaG7DvMoyUv9
From (redirected): https://drive.google.com/uc?id=1r-5t-VgDaAQidZLVgWtguaG7DvMoyUv9&confirm=t&uuid=186b8850-7a9b-4fe2-83ce-ca2018be3102
To: /content/dfmm_download/parsing.zip
100%|██████████| 94.9M/94.9M [00:00<00:00, 208MB/s]


[get ] labels.zip (575 KB) attempt 1


...

[get ] keypoints.zip (956 KB) attempt 1


Downloading...
From: https://drive.google.com/uc?id=1ZXdOQI-d4zNhqRJdUEWSQvPwAtLdjovo
To: /content/dfmm_download/keypoints.zip
100%|██████████| 979k/979k [00:00<00:00, 134MB/s]


[get ] captions.json (~11 MB) attempt 1


Downloading...
From: https://drive.google.com/uc?id=1d1TRm8UMcQhZCb6HpPo8l3OPEin4Ztk2
To: /content/dfmm_download/captions.json
100%|██████████| 12.0M/12.0M [00:00<00:00, 265MB/s]

all downloads present


## A3. Verify integrity and contents

In [4]:
report = {}
for name, (fid, approx, spec) in FILES.items():
    path = os.path.join(LOCAL_DL, name)
    r = {"bytes": os.path.getsize(path)}
    if spec.get("json"):
        data = json.load(open(path)); r["entries"] = len(data); r["ok"] = len(data) > 0
    else:
        with zipfile.ZipFile(path) as z:
            bad = z.testzip()                       # reads every member and checks its CRC
            members = [m for m in z.namelist() if not m.endswith("/")]
            n = sum(m.lower().endswith(spec["ext"]) for m in members)
            r.update(crc_ok=bad is None, first_bad_member=bad, members=len(members),
                     matching_files=n, expected=spec["count"])
            r["ok"] = (bad is None) and (n == spec["count"])
            if name == "labels.zip":
                names = {os.path.basename(m) for m in members}
                r["label_files"] = sorted(names)
    report[name] = r
    print(f"{'PASS' if r['ok'] else 'FAIL'}  {name:14s} {r}")

assert all(r["ok"] for r in report.values()), "verification failed — do not copy to Drive; re-download the FAIL files"
print("\nall files verified")

PASS  image.zip      {'bytes': 6822188210, 'crc_ok': True, 'first_bad_member': None, 'members': 44096, 'matching_files': 44096, 'expected': 44096, 'ok': True}
PASS  parsing.zip    {'bytes': 94904514, 'crc_ok': True, 'first_bad_member': None, 'members': 12701, 'matching_files': 12701, 'expected': 12701, 'ok': True}
PASS  labels.zip     {'bytes': 588902, 'crc_ok': True, 'first_bad_member': None, 'members': 3, 'matching_files': 3, 'expected': 3, 'ok': True, 'label_files': ['fabric_ann.txt', 'pattern_ann.txt', 'shape_anno_all.txt']}
PASS  keypoints.zip  {'bytes': 978992, 'crc_ok': True, 'first_bad_member': None, 'members': 2, 'matching_files': 2, 'expected': 2, 'ok': True}
PASS  captions.json  {'bytes': 11968792, 'entries': 42544, 'ok': True}

all files verified


## A4. Checksums

In [5]:
import hashlib
def sha256(path, chunk=16 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""): h.update(b)
    return h.hexdigest()

checksums = {name: {"sha256": sha256(os.path.join(LOCAL_DL, name)), **report[name]} for name in FILES}
for k, v in checksums.items(): print(f"{k:14s} {v['sha256'][:16]}…  {v['bytes']:,} bytes")

image.zip      f27aa9f1a7e024f0…  6,822,188,210 bytes
parsing.zip    8898baa9975749cb…  94,904,514 bytes
labels.zip     aa911ac1753f906d…  588,902 bytes
keypoints.zip  4a721013affc41fa…  978,992 bytes
captions.json  fffc647fc1ef8653…  11,968,792 bytes


## A5. Copy to Drive and flush

In [8]:
import shutil, json, os
from google.colab import drive

# 1. Remount so Colab sees Drive's current state (not a stale view)
drive.mount("/content/drive", force_remount=True)

# 2. Recreate the folder and prove it is writable *now*
os.makedirs(DRIVE_DIR, exist_ok=True)
probe = os.path.join(DRIVE_DIR, ".write_probe")
with open(probe, "w") as f: f.write("ok")
assert open(probe).read() == "ok"; os.remove(probe)
print("Drive folder ready:", DRIVE_DIR)

# 3. Copy each verified file: write to a temp name, check size, then rename
for name in FILES:
    src, dst = os.path.join(LOCAL_DL, name), os.path.join(DRIVE_DIR, name)
    if os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src):
        print(f"[skip] {name} already on Drive"); continue
    tmp = dst + ".partial"
    print(f"[copy] {name} ({os.path.getsize(src)/1e6:,.1f} MB) -> Drive")
    shutil.copyfile(src, tmp)
    assert os.path.getsize(tmp) == os.path.getsize(src), f"size mismatch for {name}"
    os.replace(tmp, dst)

json.dump(checksums, open(os.path.join(DRIVE_DIR, "checksums.json"), "w"), indent=2, default=str)
print("\nfiles on Drive:", sorted(os.listdir(DRIVE_DIR)))

# 4. Force the upload to finish before anything else happens
drive.flush_and_unmount()
print("Drive flushed and unmounted. Now: Runtime -> Disconnect and delete runtime, then run Section B.")

Mounted at /content/drive
Drive folder ready: /content/drive/MyDrive/hari/PRISM-GAN/datasets/DeepFashion-MultiModal/raw
[copy] image.zip (6,822.2 MB) -> Drive
[copy] parsing.zip (94.9 MB) -> Drive
[copy] labels.zip (0.6 MB) -> Drive
[copy] keypoints.zip (1.0 MB) -> Drive
[copy] captions.json (12.0 MB) -> Drive

files on Drive: ['captions.json', 'checksums.json', 'image.zip', 'keypoints.zip', 'labels.zip', 'parsing.zip']
Drive flushed and unmounted. Now: Runtime -> Disconnect and delete runtime, then run Section B.


# Section B — verify the stored copy (run in a *fresh* runtime)

*Runtime → Disconnect and delete runtime*, reconnect, then run only the cells below.
A fresh runtime has no local copy and no cache, so this proves the files really are on Drive.

In [1]:
import os, json, hashlib, zipfile
from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/hari/PRISM-GAN/datasets/DeepFashion-MultiModal/raw"

def sha256(path, chunk=16 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""): h.update(b)
    return h.hexdigest()

ref = json.load(open(os.path.join(DRIVE_DIR, "checksums.json")))
all_ok = True
for name, info in ref.items():
    p = os.path.join(DRIVE_DIR, name)
    if not os.path.exists(p):
        print(f"FAIL  {name}: missing on Drive"); all_ok = False; continue
    ok = os.path.getsize(p) == info["bytes"] and sha256(p) == info["sha256"]
    all_ok &= ok
    print(f"{'PASS' if ok else 'FAIL'}  {name:14s} {os.path.getsize(p):,} bytes")
print("\nDATASET STORED AND VERIFIED" if all_ok else "\nSOME FILES FAILED — rerun Section A for them")

Mounted at /content/drive
PASS  image.zip      6,822,188,210 bytes
PASS  parsing.zip    94,904,514 bytes
PASS  labels.zip     588,902 bytes
PASS  keypoints.zip  978,992 bytes
PASS  captions.json  11,968,792 bytes

DATASET STORED AND VERIFIED
